## Dumping and Restoring the Environment

Every LAILA policy carries configuration across communication,
command (taskforces), memory (pools), and identity.  The
`laila.environment` attribute captures all of this as a plain
JSON-serializable dictionary — the same structure that
`laila.args` accepts.

### What this notebook does
1. Create a policy with **custom communication settings**.
2. Dump the live environment to a JSON file.
3. Inspect the dump.
4. Load the dump back into `laila.args` and rebuild a policy.
5. Verify the rebuilt policy matches the original.

In [ ]:
import json
import laila
from laila.macros.defaults import DefaultPolicy, DefaultTCPIPProtocol
from dotmap import DotMap

### 1 — Create a policy with custom settings

In [ ]:
policy = DefaultPolicy()
laila.active_policy = policy

tcp = DefaultTCPIPProtocol(
    host="192.168.1.50", port=9090, peer_secret_key="my-secret-key",
)
laila.communication.add_connection(tcp)

print(f"Policy ID: {policy.global_id}")
print(f"Host:      {tcp.host}")
print(f"Port:      {tcp.port}")
print(f"Secret:    {tcp.peer_secret_key}")

### 2 — Dump the environment

In [ ]:
env = laila.environment

print(json.dumps(env, indent=2, default=str))

### 3 — Save to a JSON file

In [ ]:
with open("env_snapshot.json", "w") as f:
    json.dump(env, f, indent=2, default=str)

print("Saved to env_snapshot.json")

### 4 — Restore from the snapshot

Load the JSON file into `laila.args` and create a fresh policy.
Protocol fields live inside a `connections` dictionary keyed by
`global_id`, so they behave like pools and taskforces — they
restore exactly when the protocol is re-created with the same UUID.

In [ ]:
with open("env_snapshot.json") as f:
    loaded = json.load(f)

laila.args = DotMap(loaded)

restored = DefaultPolicy()
laila.active_policy = restored

restored_tcp = DefaultTCPIPProtocol(uuid=tcp._uuid)
laila.communication.add_connection(restored_tcp)

print(f"Restored host:   {restored_tcp.host}")
print(f"Restored port:   {restored_tcp.port}")
print(f"Restored secret: {restored_tcp.peer_secret_key}")

### 5 — Verify the match

In [ ]:
assert restored_tcp.host == "192.168.1.50"
assert restored_tcp.port == 9090
assert restored_tcp.peer_secret_key == "my-secret-key"

print("All fields restored correctly.")

### 6 — Compare the two environment dumps

In [ ]:
env_restored = laila.environment

proto_original = next(iter(env["policy"]["central"]["communication"]["connections"].values()))
proto_restored = next(iter(env_restored["policy"]["central"]["communication"]["connections"].values()))

print("Original proto:", json.dumps(proto_original, default=str))
print("Restored proto:", json.dumps(proto_restored, default=str))

assert proto_original["host"] == proto_restored["host"]
assert proto_original["port"] == proto_restored["port"]
assert proto_original["peer_secret_key"] == proto_restored["peer_secret_key"]
print("\nProtocol config matches.")

### Cleanup

In [ ]:
import os
os.remove("env_snapshot.json")
print("Cleaned up.")